# Matrix Decompositions — Exercises

8 graded exercises covering triangular solves, LU with pivoting, Householder QR, rank-revealing QR, GP regression, and differentiating through Cholesky.

| Format | Description |
|--------|-------------|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|-------|-----------|-------|
| ★ | 1–3 | Core mechanics |
| ★★ | 4–6 | Deeper theory |
| ★★★ | 7–8 | AI / ML applications |

### Topic Map

| Topic | Exercise |
|-------|----------|
| Triangular substitution | 1 |
| LU with pivoting | 2, 3 |
| Householder QR | 4, 5 |
| Rank-revealing QR | 6 |
| GP regression (Cholesky) | 7 |
| Differentiating through Cholesky | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la
import scipy.linalg as sla

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', np.asarray(expected))
        print('  got     :', np.asarray(got))
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def log_det_chol(A):
    """Compute log det A for SPD A via Cholesky."""
    L = sla.cholesky(A, lower=True)
    return 2.0 * np.sum(np.log(np.diag(L)))

print('Setup complete.')

---

## Exercise 1 (★): Forward and Backward Substitution

Triangular systems are the foundation of all matrix factorizations. This exercise implements and tests both substitution algorithms.

**Parts:**

(a) Implement `forward_sub(L, b)` for **unit** lower triangular $L$ (diagonal is all 1s — no division needed).

(b) Implement `backward_sub(U, b)` for upper triangular $U$.

(c) Verify both against `scipy.linalg.solve_triangular` on a $6 \times 6$ system.

(d) Measure the residual $\|L\hat{\mathbf{x}} - \mathbf{b}\|_\infty / \|\mathbf{b}\|_\infty$ for a $100 \times 100$ system and confirm it is $O(u) \approx 10^{-16}$.

*Hint:* Forward substitution: $x_i = b_i - \sum_{j=1}^{i-1} L_{ij} x_j$ (no division for unit $L$).
Backward substitution: $x_i = (b_i - \sum_{j=i+1}^{n} U_{ij} x_j) / U_{ii}$.

In [ ]:
# Exercise 1: Your Solution

def forward_sub(L, b):
    """Solve Lx = b for unit lower triangular L."""
    # YOUR CODE HERE
    pass

def backward_sub(U, b):
    """Solve Ux = b for upper triangular U."""
    # YOUR CODE HERE
    pass

# Given data
np.random.seed(0)
n = 6
L_test = np.tril(np.random.randn(n, n))
np.fill_diagonal(L_test, 1.0)  # unit lower triangular
U_test = np.triu(np.random.randn(n, n))
b_test = np.random.randn(n)

x_fwd = forward_sub(L_test, b_test)
x_bwd = backward_sub(U_test, b_test)
print('Forward sub result:', x_fwd)
print('Backward sub result:', x_bwd)

In [ ]:
# Exercise 1: Solution

def forward_sub(L, b):
    """Solve Lx = b for unit lower triangular L."""
    n = len(b)
    x = b.copy().astype(float)
    for i in range(1, n):
        x[i] -= L[i, :i] @ x[:i]
    return x

def backward_sub(U, b):
    """Solve Ux = b for upper triangular U."""
    n = len(b)
    x = b.copy().astype(float)
    for i in range(n-1, -1, -1):
        x[i] -= U[i, i+1:] @ x[i+1:]
        x[i] /= U[i, i]
    return x

header('Exercise 1: Forward and Backward Substitution')

np.random.seed(0)
n = 6
L_test = np.tril(np.random.randn(n, n))
np.fill_diagonal(L_test, 1.0)
U_test = np.triu(np.random.randn(n, n))
b_test = np.random.randn(n)

x_fwd = forward_sub(L_test, b_test)
x_bwd = backward_sub(U_test, b_test)

x_fwd_ref = sla.solve_triangular(L_test, b_test, lower=True, unit_diagonal=True)
x_bwd_ref = sla.solve_triangular(U_test, b_test, lower=False)

check_close('Forward sub matches scipy', x_fwd, x_fwd_ref, tol=1e-12)
check_close('Backward sub matches scipy', x_bwd, x_bwd_ref, tol=1e-12)

# Part (d): residuals for n=100
n_big = 100
L_big = np.tril(np.random.randn(n_big, n_big))
np.fill_diagonal(L_big, 1.0)
b_big = np.random.randn(n_big)
x_big = forward_sub(L_big, b_big)
res = np.linalg.norm(L_big @ x_big - b_big, np.inf) / np.linalg.norm(b_big, np.inf)
check_true(f'Residual {res:.2e} < 1e-12 (near machine epsilon)', res < 1e-12)
print(f'\nResidual: {res:.2e} (compare to u = 2.2e-16)')
print('\nTakeaway: Forward substitution is stable — residual is O(n * u), near machine epsilon.')

---

## Exercise 2 (★): LU Without Pivoting — Demonstrating Failure

Naive Gaussian elimination (without pivoting) fails catastrophically for matrices with tiny or zero leading entries, even when the system is well-conditioned.

**Parts:**

(a) Implement `lu_naive(A)` that performs Gaussian elimination without pivoting, returning $(L, U)$ with $L$ unit lower triangular.

(b) For $A = \begin{pmatrix} \varepsilon & 1 \\ 1 & 2 \end{pmatrix}$ with $\varepsilon = 10^{-16}$, $\mathbf{b} = (1, 3)^\top$: compute $\hat{\mathbf{x}}$ via naive LU and report the relative error.

(c) Compute the growth factor $\rho_2 = \max|U_{ij}| / \max|A_{ij}|$ and explain why it causes catastrophic cancellation.

(d) Show that `numpy.linalg.solve` (which uses partial pivoting) gives the correct answer.

In [ ]:
# Exercise 2: Your Solution

def lu_naive(A):
    """LU factorization WITHOUT pivoting. Returns L, U."""
    # YOUR CODE HERE
    pass

# Given data
eps = 1e-16
A_bad = np.array([[eps, 1.], [1., 2.]])
b_bad = np.array([1., 3.])

try:
    L_bad, U_bad = lu_naive(A_bad)
    print('L =', L_bad)
    print('U =', U_bad)
except Exception as e:
    print(f'Error: {e}')

In [ ]:
# Exercise 2: Solution

def lu_naive(A):
    """LU factorization WITHOUT pivoting. Returns L (unit lower tri), U."""
    n = A.shape[0]
    A = A.copy().astype(float)
    L = np.eye(n)
    for k in range(n - 1):
        if abs(A[k, k]) < 1e-300:
            raise ValueError(f'Zero pivot at step {k}')
        L[k+1:, k] = A[k+1:, k] / A[k, k]   # multipliers
        A[k+1:] -= np.outer(L[k+1:, k], A[k])
    return L, np.triu(A)

header('Exercise 2: LU Failure Without Pivoting')

eps = 1e-16
A_bad = np.array([[eps, 1.], [1., 2.]])
b_bad = np.array([1., 3.])

# True solution
x_true = np.linalg.solve(A_bad, b_bad)
print(f'True solution: {x_true}')

# Naive LU solution
L_bad, U_bad = lu_naive(A_bad)
y_bad = forward_sub(L_bad, b_bad)
x_naive = backward_sub(U_bad, y_bad)
rel_err = np.linalg.norm(x_naive - x_true) / np.linalg.norm(x_true)
print(f'Naive LU solution: {x_naive}')

# Growth factor
rho2 = np.abs(U_bad).max() / np.abs(A_bad).max()
check_true(f'Growth factor rho2 = {rho2:.2e} >> 1 (catastrophic)', rho2 > 1e10)
check_true(f'Naive LU relative error = {rel_err:.2e} > 0.1 (large)', rel_err > 0.1)

# Partial pivoting (numpy.linalg.solve)
x_numpy = np.linalg.solve(A_bad, b_bad)
err_numpy = np.linalg.norm(x_numpy - x_true) / np.linalg.norm(x_true)
check_true(f'numpy.linalg.solve error = {err_numpy:.2e} < 1e-10 (correct)', err_numpy < 1e-10)

print(f'\nNaive LU error: {rel_err:.2e} (useless)')
print(f'numpy solve error: {err_numpy:.2e} (machine epsilon)')
print('\nTakeaway: Never use LU without pivoting. Partial pivoting ensures |L_ij| <= 1,')
print('bounding the growth factor in practice to O(n^{2/3}) for typical inputs.')

---

## Exercise 3 (★): LU with Partial Pivoting — Verify PA = LU

Implement LU with partial pivoting and verify the factorization identity $PA = LU$.

**Parts:**

(a) Implement `lu_pivot(A)` returning `(piv, L, U)` where `piv` is a permutation array (not a matrix): `A[piv, :] = L @ U`.

(b) Verify $\|A[\texttt{piv}, :] - LU\|_F \leq c \cdot n \cdot u \cdot \|A\|_F$ on a $40 \times 40$ random matrix.

(c) Compute $\det(A)$ from the LU factorization as $(-1)^s \prod_i U_{ii}$ and verify against `numpy.linalg.det`.

(d) Solve $A\mathbf{x} = \mathbf{b}$ using your factorization and compare accuracy to `numpy.linalg.solve`.

In [ ]:
# Exercise 3: Your Solution

def lu_pivot(A):
    """LU with partial pivoting. Returns (piv, L, U) where A[piv,:] = L @ U."""
    # YOUR CODE HERE
    pass

# Given data
np.random.seed(1)
n_ex3 = 40
A_ex3 = np.random.randn(n_ex3, n_ex3)
b_ex3 = np.random.randn(n_ex3)

result = lu_pivot(A_ex3)
print('lu_pivot result:', result)

In [ ]:
# Exercise 3: Solution

def lu_pivot(A):
    """LU with partial pivoting. Returns (piv, L, U) where A[piv,:] = L @ U."""
    n = A.shape[0]
    A = A.copy().astype(float)
    piv = np.arange(n)
    L = np.zeros((n, n))
    np.fill_diagonal(L, 1.0)
    n_swaps = 0

    for k in range(n - 1):
        p = k + np.argmax(np.abs(A[k:, k]))
        if p != k:
            A[[k, p]] = A[[p, k]]
            L[[k, p], :k] = L[[p, k], :k]
            piv[[k, p]] = piv[[p, k]]
            n_swaps += 1
        if abs(A[k, k]) > 1e-14:
            L[k+1:, k] = A[k+1:, k] / A[k, k]
            A[k+1:] -= np.outer(L[k+1:, k], A[k])

    U = np.triu(A)
    return piv, L, U, n_swaps

header('Exercise 3: LU with Partial Pivoting')

np.random.seed(1)
n_ex3 = 40
A_ex3 = np.random.randn(n_ex3, n_ex3)
b_ex3 = np.random.randn(n_ex3)

piv_ex3, L_ex3, U_ex3, n_swaps = lu_pivot(A_ex3)

# (b) Verify PA = LU
err_LU = np.linalg.norm(A_ex3[piv_ex3] - L_ex3 @ U_ex3, 'fro')
expected_bound = n_ex3 * np.finfo(float).eps * np.linalg.norm(A_ex3, 'fro')
check_true(f'||PA - LU||_F = {err_LU:.2e} < {expected_bound:.2e}', err_LU < expected_bound * 10)

# (c) Determinant via LU
sign = (-1)**n_swaps
det_lu_ex3 = sign * np.prod(np.diag(U_ex3))
det_np_ex3 = np.linalg.det(A_ex3)
check_close('det(LU) vs numpy.linalg.det', det_lu_ex3, det_np_ex3, tol=1e-6)

# (d) Solve Ax = b
x_true_ex3 = np.linalg.solve(A_ex3, b_ex3)
b_perm = b_ex3[piv_ex3]
y_ex3 = forward_sub(L_ex3, b_perm)
x_ex3 = backward_sub(U_ex3, y_ex3)
err_x = np.linalg.norm(x_ex3 - x_true_ex3) / np.linalg.norm(x_true_ex3)
check_true(f'Solution error = {err_x:.2e} < 1e-10', err_x < 1e-10)

print('\nTakeaway: Partial pivoting ensures |L_ij| <= 1, making the factorization')
print('numerically stable in practice. det(A) = (-1)^s * prod(diag(U)).')

---

## Exercise 4 (★★): Householder Reflector Construction

Implement a Householder reflector $H = I - 2\mathbf{v}\mathbf{v}^\top/\|\mathbf{v}\|^2$ and verify its action and orthogonality.

**Parts:**

(a) Implement `householder_vector(a)` computing the Householder vector $\mathbf{v}$ with the correct sign convention: $\alpha = -\operatorname{sign}(a_0)\|\mathbf{a}\|$.

(b) Implement `householder_apply(v, beta, C)` that applies $H$ to $C$ implicitly using $HC = C - \beta \mathbf{v}(\mathbf{v}^\top C)$.

(c) Verify: for $\mathbf{a} = (3, 1, 4, 1, 5)^\top$, confirm $H\mathbf{a} = \alpha\mathbf{e}_1$ with entries $2$–$5$ equal to zero (to $10^{-14}$).

(d) Verify $H^2 = I$ (involutory) and $HH^\top = I$ (orthogonal) for a $5 \times 5$ $H$.

(e) For an ill-conditioned matrix (Hilbert $6\times 6$), compare $\|\hat{Q}^\top\hat{Q} - I\|_F$ from classical Gram-Schmidt vs Householder QR.

*Sign convention:* Choose $\alpha = -\operatorname{sign}(a_0)\|\mathbf{a}\|$ to avoid cancellation in $v_0 = a_0 - \alpha$.

In [ ]:
# Exercise 4: Your Solution

def householder_vector(a):
    """Compute (v, beta) such that (I - beta*v*v^T) @ a = alpha * e_1."""
    # YOUR CODE HERE
    pass

def householder_apply(v, beta, C):
    """Apply H = I - beta*v*v^T to C implicitly."""
    # YOUR CODE HERE
    pass

import numpy as np
a_test = np.array([3., 1., 4., 1., 5.])
result = householder_vector(a_test)
print('householder_vector result:', result)

In [ ]:
# Exercise 4: Solution

import numpy as np, scipy.linalg as sla

def householder_vector(a):
    """Compute (v, beta) such that (I - beta*v*v^T) @ a = alpha * e_1.
    Sign convention: alpha = -sign(a[0]) * norm(a) avoids cancellation."""
    n = len(a)
    sigma = np.linalg.norm(a)
    if sigma == 0:
        return np.zeros(n), 0.0
    alpha = -np.sign(a[0]) * sigma
    v = a.copy().astype(float)
    v[0] -= alpha          # v = a - alpha * e_1
    beta = 2.0 / (v @ v)  # 2 / ||v||^2
    return v, beta

def householder_apply(v, beta, C):
    """Apply H = I - beta*v*v^T to C implicitly: C -> C - beta*v*(v^T C)."""
    w = beta * (v @ C)   # scalar or row vector
    return C - np.outer(v, w)

header('Exercise 4: Householder Reflector')

# (c) Verify H @ a = alpha * e_1
a_test = np.array([3., 1., 4., 1., 5.])
v_h, beta_h = householder_vector(a_test)
Ha = a_test - beta_h * v_h * (v_h @ a_test)

alpha_expected = -np.sign(a_test[0]) * np.linalg.norm(a_test)
check_true(f'Ha[0] = alpha = {Ha[0]:.4f} (expected {alpha_expected:.4f})',
           abs(Ha[0] - alpha_expected) < 1e-12)
check_true('Ha[1:] = 0 (entries 2-5 zeroed out)', np.allclose(Ha[1:], 0, atol=1e-14))

# (d) H^2 = I and H H^T = I
n_h = 5
H_mat = np.eye(n_h) - beta_h * np.outer(v_h, v_h)
check_close('H^2 = I (involutory)', H_mat @ H_mat, np.eye(n_h))
check_close('H H^T = I (orthogonal)', H_mat @ H_mat.T, np.eye(n_h))

# (e) Gram-Schmidt vs Householder on Hilbert matrix
n_hilb = 6
H_hilb = np.array([[1.0/(i+j+1) for j in range(n_hilb)] for i in range(n_hilb)])
kappa_hilb = np.linalg.cond(H_hilb)
print(f'\nHilbert matrix condition number: {kappa_hilb:.2e}')

# Classical Gram-Schmidt
Q_cgs = np.zeros((n_hilb, n_hilb))
for j in range(n_hilb):
    v = H_hilb[:, j].astype(float)
    for i in range(j):
        v -= (Q_cgs[:, i] @ H_hilb[:, j]) * Q_cgs[:, i]
    Q_cgs[:, j] = v / np.linalg.norm(v)

Q_house_ref, _ = np.linalg.qr(H_hilb)

orth_cgs = np.linalg.norm(Q_cgs.T @ Q_cgs - np.eye(n_hilb), 'fro')
orth_house = np.linalg.norm(Q_house_ref.T @ Q_house_ref - np.eye(n_hilb), 'fro')

print(f'Classical GS orthogonality error:  {orth_cgs:.2e}')
print(f'Householder orthogonality error:   {orth_house:.2e}')
check_true('Householder is more orthogonal than classical GS', orth_house < orth_cgs)

print('\nTakeaway: Householder QR maintains ||Q^T Q - I||_F = O(u) regardless of')
print('condition number. Classical GS has error O(kappa * u).')

---

## Exercise 5 (★★): Householder QR Algorithm

Implement the full Householder QR factorization and use it to solve a least-squares problem.

**Parts:**

(a) Implement `householder_qr(A)` returning thin $(Q, R)$ where $Q \in \mathbb{R}^{m \times n}$ and $R \in \mathbb{R}^{n \times n}$.

(b) Verify $\|A - QR\|_F \leq c \cdot u \cdot \|A\|_F$ and $\|Q^\top Q - I\|_F \leq c \cdot u$ for a $50 \times 30$ random matrix.

(c) Use your QR to solve the least-squares problem $\min \|A\mathbf{x} - \mathbf{b}\|_2$ via $R\mathbf{x} = Q^\top\mathbf{b}$ and compare to `numpy.linalg.lstsq`.

(d) Compare condition number loss: form $A^\top A$ and measure $\kappa(A^\top A)$ vs $\kappa(A)$. Confirm $\kappa(A^\top A) \approx \kappa(A)^2$.

In [ ]:
# Exercise 5: Your Solution

def householder_qr(A):
    """Compute thin QR via Householder reflectors. Returns (Q, R)."""
    # YOUR CODE HERE
    pass

import numpy as np
np.random.seed(2)
m_e5, n_e5 = 50, 30
A_e5 = np.random.randn(m_e5, n_e5)

result = householder_qr(A_e5)
print('householder_qr result shapes:', [r.shape if hasattr(r, 'shape') else r for r in (result or [])])

In [ ]:
# Exercise 5: Solution

import numpy as np, scipy.linalg as sla

def householder_qr(A):
    """Compute thin QR via Householder. Returns (Q, R) with Q m x n, R n x n."""
    m, n = A.shape
    A_work = A.copy().astype(float)
    reflectors = []

    for k in range(n):
        a_col = A_work[k:, k]
        sigma = np.linalg.norm(a_col)
        if sigma < 1e-14:
            reflectors.append((k, None, 0.0))
            continue
        alpha = -np.sign(a_col[0]) * sigma
        v = a_col.copy()
        v[0] -= alpha
        beta = 2.0 / (v @ v)
        reflectors.append((k, v, beta))
        # Apply H_k implicitly
        w = beta * (v @ A_work[k:, k:])
        A_work[k:, k:] -= np.outer(v, w)

    R = np.triu(A_work[:n, :])

    # Build Q by applying reflectors in reverse
    Q = np.eye(m, n)
    for k, v, beta in reversed(reflectors):
        if v is None: continue
        w = beta * (v @ Q[k:, k:])
        Q[k:, k:] -= np.outer(v, w)

    return Q, R

header('Exercise 5: Householder QR')

np.random.seed(2)
m_e5, n_e5 = 50, 30
A_e5 = np.random.randn(m_e5, n_e5)
b_e5 = np.random.randn(m_e5)

Q_e5, R_e5 = householder_qr(A_e5)

# (b) Verify factorization accuracy and orthogonality
err_fac = np.linalg.norm(A_e5 - Q_e5 @ R_e5, 'fro')
err_orth = np.linalg.norm(Q_e5.T @ Q_e5 - np.eye(n_e5), 'fro')
bound_fac = 10 * m_e5 * np.finfo(float).eps * np.linalg.norm(A_e5, 'fro')
check_true(f'||A - QR||_F = {err_fac:.2e} < {bound_fac:.2e}', err_fac < bound_fac)
check_true(f'||Q^TQ - I||_F = {err_orth:.2e} < 1e-12', err_orth < 1e-12)

# (c) Least squares via QR
x_qr_e5 = sla.solve_triangular(R_e5, Q_e5.T @ b_e5, lower=False)
x_lstsq, _, _, _ = np.linalg.lstsq(A_e5, b_e5, rcond=None)
check_close('QR LS solution vs numpy lstsq', x_qr_e5, x_lstsq, tol=1e-8)

# (d) Condition number squaring in normal equations
kappa_A = np.linalg.cond(A_e5)
kappa_ATA = np.linalg.cond(A_e5.T @ A_e5)
print(f'\nkappa(A) = {kappa_A:.2e}')
print(f'kappa(A^T A) = {kappa_ATA:.2e}')
print(f'kappa(A^T A) / kappa(A)^2 = {kappa_ATA / kappa_A**2:.2f} (should be ~1)')
check_true('kappa(A^T A) approx kappa(A)^2', abs(kappa_ATA / kappa_A**2 - 1) < 0.5)

print('\nTakeaway: QR maintains condition number kappa(A) throughout;')
print('normal equations amplify to kappa(A)^2 — twice the digits lost.')

---

## Exercise 6 (★★): Rank Estimation via Column-Pivoted QR

Rank-revealing QR (RRQR) reorders columns to expose the rank structure in the diagonal of $R$. This exercise uses RRQR to estimate the numerical rank of a near-rank-deficient matrix.

**Parts:**

(a) Build a matrix of known rank 4: $A = U S V^\top$ where $S = \operatorname{diag}(100, 10, 1, 0.1, 10^{-3}, 10^{-3}, 10^{-3}, 10^{-3})$ and $U, V$ are random orthogonal.

(b) Compute column-pivoted QR via `scipy.linalg.qr(A, pivoting=True)`. Print the diagonal of $|R|$ and identify where the gap occurs.

(c) Implement rank estimation: declare rank $r$ where $|R_{rr}| / |R_{11}| > \tau$ for threshold $\tau = 10^{-2}$.

(d) Compare to SVD-based rank estimation. Which is more reliable? Which is cheaper? Report the ratio of condition numbers: $|R_{11}| / |R_{55}|$ vs $\sigma_1 / \sigma_5$.

In [ ]:
# Exercise 6: Your Solution

import numpy as np, scipy.linalg as sla

# (a) Build the matrix
np.random.seed(3)
m6, n6 = 20, 8
# YOUR CODE HERE: build A with known rank 4
A_e6 = None

# (b) Column-pivoted QR
# Q6, R6, P6 = sla.qr(A_e6, pivoting=True)
# YOUR CODE HERE

print('A_e6 shape:', A_e6)

In [ ]:
# Exercise 6: Solution

import numpy as np, scipy.linalg as sla

header('Exercise 6: Rank Estimation via RRQR')

# (a) Build known rank-4 matrix
np.random.seed(3)
m6, n6 = 20, 8
U6 = np.linalg.qr(np.random.randn(m6, m6))[0][:, :n6]
V6 = np.linalg.qr(np.random.randn(n6, n6))[0]
sv6 = np.array([100., 10., 1., 0.1, 1e-3, 1e-3, 1e-3, 1e-3])
S6 = np.diag(sv6)
A_e6 = U6 @ S6 @ V6.T
print(f'A shape: {A_e6.shape}, true singular values: {sv6}')

# (b) Column-pivoted QR
Q6, R6, P6 = sla.qr(A_e6, pivoting=True)
R_diag6 = np.abs(np.diag(R6))
print('\n|R| diagonal:')
for i, rd in enumerate(R_diag6):
    print(f'  |R[{i},{i}]| = {rd:.4f}  (ratio = {rd/R_diag6[0]:.2e})')

# (c) Rank estimation
tau = 1e-2
rank_rrqr = int(np.sum(R_diag6 / R_diag6[0] > tau))

# SVD rank estimation
sv_actual = np.linalg.svd(A_e6, compute_uv=False)
rank_svd = int(np.sum(sv_actual / sv_actual[0] > tau))

check_true(f'RRQR rank estimate = {rank_rrqr} (expect 4)', rank_rrqr == 4)
check_true(f'SVD rank estimate = {rank_svd} (expect 4)', rank_svd == 4)

# (d) Comparison
ratio_R = R_diag6[0] / R_diag6[4]  # ratio at rank gap
ratio_sv = sv_actual[0] / sv_actual[4]
print(f'\n|R[0,0]|/|R[4,4]| = {ratio_R:.2e} (RRQR gap)')
print(f'sigma_1/sigma_5   = {ratio_sv:.2e} (SVD gap)')
print(f'These ratios should be similar: {abs(np.log10(ratio_R) - np.log10(ratio_sv)):.1f} decades apart')

print('\nTakeaway: RRQR gives reliable rank estimates at O(mn^2) cost —')
print('same as full QR. SVD is also O(mn^2+n^3) but gives exact singular values.')
print('RRQR is preferred when rank estimation is needed during QR computation.')

---

## Exercise 7 (★★★): GP Regression via Cholesky

Gaussian process regression requires Cholesky factorization for all linear solves and log-determinant computation. This exercise implements the complete pipeline.

**Parts:**

(a) Implement `rbf_kernel(X1, X2, ell, sf)` computing the RBF (squared exponential) kernel $k(x, x') = s_f^2 \exp(-\|x - x'\|^2 / (2\ell^2))$.

(b) Generate 60 training points from $f(x) = \sin(2\pi x / 5)$ on $[0, 10]$ with $\mathcal{N}(0, 0.15^2)$ noise.

(c) Implement `gp_solve(K_noisy, y)` using Cholesky factorization to compute $\boldsymbol{\alpha} = (K + \sigma_n^2 I)^{-1}\mathbf{y}$ and $\log p(\mathbf{y}) = -\frac{1}{2}\mathbf{y}^\top\boldsymbol{\alpha} - \sum_i\log L_{ii} - \frac{n}{2}\log 2\pi$.

(d) Verify: $\|(K + \sigma^2 I)\boldsymbol{\alpha} - \mathbf{y}\|_\infty / \|\mathbf{y}\|_\infty < 10^{-10}$.

(e) Grid-search the length-scale $\ell \in [0.5, 5]$ to maximize the log marginal likelihood. Report the optimal $\ell$ and plot the posterior mean.

In [ ]:
# Exercise 7: Your Solution

import numpy as np, scipy.linalg as sla

def rbf_kernel(X1, X2, ell=1.0, sf=1.0):
    """RBF kernel."""
    # YOUR CODE HERE
    pass

def gp_solve(K_noisy, y):
    """Returns (alpha, log_marg_lik) using Cholesky."""
    # YOUR CODE HERE
    pass

# Given data
np.random.seed(7)
n_e7 = 60
X_e7 = np.linspace(0, 10, n_e7).reshape(-1, 1)
y_e7 = np.sin(2 * np.pi * X_e7.ravel() / 5) + 0.15 * np.random.randn(n_e7)

K_test = rbf_kernel(X_e7, X_e7, ell=1.0, sf=1.0)
print('Kernel shape:', K_test.shape if K_test is not None else None)

In [ ]:
# Exercise 7: Solution

import numpy as np, scipy.linalg as sla

def rbf_kernel(X1, X2, ell=1.0, sf=1.0):
    """RBF/SE kernel: sf^2 * exp(-||x1-x2||^2 / (2*ell^2))."""
    X1 = np.atleast_2d(X1)
    X2 = np.atleast_2d(X2)
    diff = X1[:, None, :] - X2[None, :, :]
    sq_dist = np.sum(diff**2, axis=-1)
    return sf**2 * np.exp(-sq_dist / (2 * ell**2))

def gp_solve(K_noisy, y):
    """Cholesky-based GP solve. Returns (alpha, log_marg_lik)."""
    n = len(y)
    L = sla.cholesky(K_noisy, lower=True)
    v = sla.solve_triangular(L, y, lower=True)
    alpha = sla.solve_triangular(L.T, v, lower=False)
    log_det = 2 * np.sum(np.log(np.diag(L)))
    log_marg_lik = -0.5 * (y @ alpha) - 0.5 * log_det - 0.5 * n * np.log(2 * np.pi)
    return alpha, log_marg_lik

header('Exercise 7: GP Regression via Cholesky')

np.random.seed(7)
n_e7 = 60
X_e7 = np.linspace(0, 10, n_e7).reshape(-1, 1)
y_e7 = np.sin(2 * np.pi * X_e7.ravel() / 5) + 0.15 * np.random.randn(n_e7)
sigma_n_e7 = 0.15

# (d) Verify Cholesky solve residual
ell_test = 2.0
K_nn = rbf_kernel(X_e7, X_e7, ell=ell_test, sf=1.0)
K_noisy_e7 = K_nn + sigma_n_e7**2 * np.eye(n_e7)
alpha_e7, log_lik_e7 = gp_solve(K_noisy_e7, y_e7)

residual = np.linalg.norm(K_noisy_e7 @ alpha_e7 - y_e7, np.inf) / np.linalg.norm(y_e7, np.inf)
check_true(f'Residual = {residual:.2e} < 1e-10', residual < 1e-10)
print(f'Log marginal likelihood at ell={ell_test}: {log_lik_e7:.3f}')

# (e) Grid search over ell
ell_grid = np.linspace(0.5, 5, 50)
log_liks_e7 = []
for ell in ell_grid:
    K = rbf_kernel(X_e7, X_e7, ell=ell, sf=1.0) + sigma_n_e7**2 * np.eye(n_e7)
    try:
        _, ll = gp_solve(K, y_e7)
        log_liks_e7.append(ll)
    except Exception:
        log_liks_e7.append(-np.inf)

best_ell_e7 = ell_grid[np.argmax(log_liks_e7)]
print(f'Optimal ell = {best_ell_e7:.2f}')
check_true('Optimal ell in range [1, 4] (reasonable for sin(2pi*x/5))', 0.5 < best_ell_e7 < 5)

print('\nTakeaway: Cholesky is the computational backbone of GP regression.')
print('Cost: O(n^3) for factorization, O(n^2) for each query.')
print('Log marginal likelihood = data fit term - complexity penalty (log det).')

---

## Exercise 8 (★★★): Differentiating Through Cholesky

Modern probabilistic ML requires gradients through Cholesky factorizations. This exercise implements the gradient of $\log\det A$ w.r.t. $A$ using implicit differentiation.

**Parts:**

(a) For an SPD matrix $A$, the analytical gradient is $\nabla_A \log\det A = A^{-1}$. Verify this numerically via finite differences for a $6 \times 6$ SPD matrix.

(b) Implement `chol_log_det_grad(A)` that computes $\nabla_A \log\det A = A^{-1}$ via Cholesky (without explicitly forming $A^{-1}$ via `inv` — use `cho_solve`).

(c) Implement gradient descent on $\mathcal{L}(A) = -\log\det A + \operatorname{tr}(A)$ (Gaussian log-likelihood with $\mathbf{y} = \mathbf{0}$) starting from $A_0 = 3I$. Verify convergence to $A^* = I$ in 50 steps with step size $\eta = 0.1$.

(d) Verify your gradient matches PyTorch's autograd (if available) or the finite difference gradient to relative error $< 10^{-4}$.

*Theory:* $\nabla_A \log\det A = A^{-1}$ follows from $\delta \log\det A = \operatorname{tr}(A^{-1} \delta A)$.
The gradient of $\operatorname{tr}(A) = \sum_i A_{ii}$ is the identity matrix $I$.
So $\nabla_A \mathcal{L} = -A^{-1} + I$, minimized at $A^{-1} = I$, i.e., $A = I$.

In [ ]:
# Exercise 8: Your Solution

import numpy as np, scipy.linalg as sla

def chol_log_det_grad(A):
    """Compute gradient of log det A = A^{-1} via Cholesky (no explicit inv)."""
    # YOUR CODE HERE
    pass

# Given data
np.random.seed(8)
n_e8 = 6
X8 = np.random.randn(n_e8, n_e8 + 3)
A_e8 = X8 @ X8.T / (n_e8 + 3) + 0.5 * np.eye(n_e8)

grad_result = chol_log_det_grad(A_e8)
print('Gradient shape:', grad_result.shape if grad_result is not None else None)

In [ ]:
# Exercise 8: Solution

import numpy as np, scipy.linalg as sla

def chol_log_det_grad(A):
    """Gradient of log det A = A^{-1}, computed via Cholesky cho_solve."""
    L = sla.cholesky(A, lower=True)
    A_inv = sla.cho_solve((L, True), np.eye(A.shape[0]))
    return A_inv

header('Exercise 8: Differentiating Through Cholesky')

np.random.seed(8)
n_e8 = 6
X8 = np.random.randn(n_e8, n_e8 + 3)
A_e8 = X8 @ X8.T / (n_e8 + 3) + 0.5 * np.eye(n_e8)

# (a) Finite difference verification
eps_fd = 1e-5
grad_fd_e8 = np.zeros((n_e8, n_e8))
for i in range(n_e8):
    for j in range(n_e8):
        A_p = A_e8.copy(); A_p[i,j] += eps_fd
        A_m = A_e8.copy(); A_m[i,j] -= eps_fd
        ld_p = log_det_chol(A_p)
        ld_m = log_det_chol(A_m)
        grad_fd_e8[i,j] = (ld_p - ld_m) / (2 * eps_fd)

# Symmetrize (log det is a function of symmetric A)
grad_fd_sym = (grad_fd_e8 + grad_fd_e8.T) / 2

# (b) Cholesky-based gradient
grad_chol_e8 = chol_log_det_grad(A_e8)

err_grad_e8 = np.linalg.norm(grad_chol_e8 - grad_fd_sym) / np.linalg.norm(grad_fd_sym)
check_true(f'Gradient error = {err_grad_e8:.2e} < 1e-4', err_grad_e8 < 1e-4)

# Verify gradient equals A^{-1}
A_inv_e8 = np.linalg.inv(A_e8)
err_inv_e8 = np.linalg.norm(grad_chol_e8 - A_inv_e8) / np.linalg.norm(A_inv_e8)
check_close('grad log det A = A^{-1}', grad_chol_e8, A_inv_e8, tol=1e-10)

# (c) Gradient descent on L(A) = -log det A + tr(A)
# Gradient: -A^{-1} + I. Minimum at A = I.
A_curr = 3 * np.eye(n_e8)  # start from 3I
eta = 0.1
print(f'\nGradient descent on L(A) = -log det A + tr(A):')
print(f'Starting from A_0 = 3I, expect convergence to A* = I')

loss_history = []
for step in range(60):
    # Loss
    L_curr = sla.cholesky(A_curr, lower=True)
    loss = -2*np.sum(np.log(np.diag(L_curr))) + np.trace(A_curr)
    loss_history.append(loss)
    # Gradient: -A^{-1} + I
    A_inv_curr = sla.cho_solve((L_curr, True), np.eye(n_e8))
    grad = -A_inv_curr + np.eye(n_e8)
    A_curr = A_curr - eta * grad
    # Project onto SPD cone (ensure symmetry)
    A_curr = (A_curr + A_curr.T) / 2
    A_curr += max(0, -np.linalg.eigvalsh(A_curr).min() + 1e-6) * np.eye(n_e8)

err_to_I = np.linalg.norm(A_curr - np.eye(n_e8), 'fro')
check_true(f'||A_final - I||_F = {err_to_I:.4f} < 0.1 (converged to I)', err_to_I < 0.1)
print(f'Loss at convergence: {loss_history[-1]:.4f} (expect -n = {-n_e8})')

print('\nTakeaway: Gradients through Cholesky use implicit differentiation:')
print('d(log det A)/dA = A^{-1}, computed via two triangular solves — O(n^2) cost.')
print('This is used in GP hyperparameter optimization and normalizing flows.')

---

## What to Review After Finishing

- [ ] **Triangular substitution** — forward (unit L) and backward (upper U) — $O(n^2)$
- [ ] **LU failure** — why zero/small pivots cause catastrophic growth factor
- [ ] **Partial pivoting** — $|L_{ij}| \leq 1$, growth factor bounded in practice
- [ ] **Householder sign convention** — $\alpha = -\operatorname{sign}(a_0)\|\mathbf{a}\|$ avoids cancellation
- [ ] **QR for least squares** — $\kappa(A)$ vs $\kappa(A^\top A) = \kappa(A)^2$
- [ ] **RRQR rank gap** — where $|R_{kk}|/|R_{11}|$ drops significantly
- [ ] **GP Cholesky pipeline** — build $K$, factor $L$, solve $(K+\sigma^2 I)\alpha = y$, compute $\log\det$
- [ ] **Gradient of log det** — $\nabla_A \log\det A = A^{-1}$, compute via `cho_solve`

## References

1. Golub & Van Loan (2013). *Matrix Computations* (4th ed.). §3 (LU), §5 (QR).
2. Trefethen & Bau (1997). *Numerical Linear Algebra*. Lectures 16–21 (QR, stability).
3. Higham (2002). *Accuracy and Stability*. Ch. 9 (LU growth factors), Ch. 19 (QR stability).
4. Martens & Grosse (2015). *K-FAC*. ICML 2015.
5. Halko, Martinsson & Tropp (2011). *Randomized matrix decompositions*. SIAM Review.
6. Gardner et al. (2018). *GPyTorch*. NeurIPS 2018.
7. Hu et al. (2022). *LoRA*. ICLR 2022.
